In [35]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
#biblio pour scaler 
from sklearn.preprocessing import StandardScaler
#modèle d'entrainement
from sklearn.ensemble import RandomForestClassifier

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

#reading the data
df_train = pd.read_csv("/kaggle/input/titanic/train.csv")
#Xtrain = DataFrame entrainer le modèle avec Xtrain et Ytrain
X_train = df_train.drop(['Survived'], axis=1)
#Xtest = DataFrame pour faire prédiction de Ytest
X_test = pd.read_csv("/kaggle/input/titanic/test.csv")
#Ytrain = Resultat si survie ou pas 
y_train = df_train["Survived"]

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


/kaggle/input/titanic/train.csv
/kaggle/input/titanic/test.csv
/kaggle/input/titanic/gender_submission.csv


In [36]:
print("Aperçu des premières lignes de X_train:")
print(X_train.head())  # Affiche les premières lignes de X_train
print("\nAperçu des premières lignes de X_test:")
print(X_test.head())  # Affiche les premières lignes de X_test

print("\nInformations sur X_train:")
print(X_train.info())  # Affiche le résumé des informations de X_train
print("\nInformations sur X_test:")
print(X_test.info())  # Affiche le résumé des informations de X_test

Aperçu des premières lignes de X_train:
   PassengerId  Pclass                                               Name  \
0            1       3                            Braund, Mr. Owen Harris   
1            2       1  Cumings, Mrs. John Bradley (Florence Briggs Th...   
2            3       3                             Heikkinen, Miss. Laina   
3            4       1       Futrelle, Mrs. Jacques Heath (Lily May Peel)   
4            5       3                           Allen, Mr. William Henry   

      Sex   Age  SibSp  Parch            Ticket     Fare Cabin Embarked  
0    male  22.0      1      0         A/5 21171   7.2500   NaN        S  
1  female  38.0      1      0          PC 17599  71.2833   C85        C  
2  female  26.0      0      0  STON/O2. 3101282   7.9250   NaN        S  
3  female  35.0      1      0            113803  53.1000  C123        S  
4    male  35.0      0      0            373450   8.0500   NaN        S  

Aperçu des premières lignes de X_test:
   PassengerI

In [37]:
#Pour affiner les prédictions on va dans un premier temps dégager les infos inutiles comme le nom ou les num des cabines en plus plein de valeur manquante
X_train = X_train.drop(["Name", "Ticket", "Cabin"], axis=1)
X_test = X_test.drop(["Name", "Ticket", "Cabin"], axis=1)

In [38]:
print("Aperçu des premières lignes de X_train:")
print(X_train.head())  # Affiche les premières lignes de X_train
print("\nAperçu des premières lignes de X_test:")
print(X_test.head())  # Affiche les premières lignes de X_test
#Name ticket et cabin ont bien été enlevé

Aperçu des premières lignes de X_train:
   PassengerId  Pclass     Sex   Age  SibSp  Parch     Fare Embarked
0            1       3    male  22.0      1      0   7.2500        S
1            2       1  female  38.0      1      0  71.2833        C
2            3       3  female  26.0      0      0   7.9250        S
3            4       1  female  35.0      1      0  53.1000        S
4            5       3    male  35.0      0      0   8.0500        S

Aperçu des premières lignes de X_test:
   PassengerId  Pclass     Sex   Age  SibSp  Parch     Fare Embarked
0          892       3    male  34.5      0      0   7.8292        Q
1          893       3  female  47.0      1      0   7.0000        S
2          894       2    male  62.0      0      0   9.6875        Q
3          895       3    male  27.0      0      0   8.6625        S
4          896       3  female  22.0      1      1  12.2875        S


In [39]:
#en regardant la data frame, on peut voir pas mal de valeur manquante
# on remplit par la moyenne car il y a des donné manquante pour ces catégories dans ces dataframes
X_train["Age"] = X_train["Age"].fillna(X_train["Age"].median())
X_test["Age"] = X_test["Age"].fillna(X_test["Age"].median())
X_test["Fare"] = X_test["Fare"].fillna(X_test["Fare"].median())

# Remplir les valeurs manquantes pour 'Embarked' avec la valeur la plus fréquente

X_train["Embarked"] = X_train["Embarked"].fillna("S")


In [40]:
print("\nInformations sur X_train:")
print(X_train.info())  # Affiche le résumé des informations de X_train
print("\nInformations sur X_test:")
print(X_test.info())  # Affiche le résumé des informations de X_test
#on a bien rempli les données vides 891/891 pour Xtrain et 418 pour Xtest


Informations sur X_train:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Pclass       891 non-null    int64  
 2   Sex          891 non-null    object 
 3   Age          891 non-null    float64
 4   SibSp        891 non-null    int64  
 5   Parch        891 non-null    int64  
 6   Fare         891 non-null    float64
 7   Embarked     891 non-null    object 
dtypes: float64(2), int64(4), object(2)
memory usage: 55.8+ KB
None

Informations sur X_test:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Sex          418 non-null    object 
 3   Age          418 non-null    float64

In [ ]:
#Le modèle ne comprend pas les lettres on doit alors essayer de trouver un moyen pour qu'il comprenne chaque donnée
#D'abord pour le sexe on défini male et female en 0 et 1 
X_train["Sex"] = X_train["Sex"].map({"male": 0, "female": 1})
X_test["Sex"] = X_test["Sex"].map({"male": 0, "female": 1})

#Pour la catégorie embarked il y a 3 valeurs "Q,S,C" et comparer au sexe on peut pas faire 0 1 2 car sinon il y aurait un certain ordre qui se crée
# On crée alors un tableau en binaire qui correspond à 0 1 2 
X_train = pd.get_dummies(X_train, columns=["Embarked"], drop_first=True)
X_test = pd.get_dummies(X_test, columns=["Embarked"], drop_first=True)

#on va s'occuper du scaling sur les données qui ont trop grande plage comme Fare = tarif et Age 
#on "standardise" les valeurs pour accélérer l’apprentissage et améliorer les performances.
#les colonnes numériques ont une échelle similaire
scaler = StandardScaler()
X_train[["Age", "Fare"]] = scaler.fit_transform(X_train[["Age", "Fare"]])
X_test[["Age", "Fare"]] = scaler.transform(X_test[["Age", "Fare"]])

In [42]:
print("\nInformations sur X_train:")
print(X_train.info())  # Affiche le résumé des informations de X_train
print("\nInformations sur X_test:")
print(X_test.info())  # Affiche le résumé des informations de X_test
#il n'y a plus de donnée manquante et le valeur peuvent être lu par le modèle 


Informations sur X_train:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Pclass       891 non-null    int64  
 2   Sex          891 non-null    int64  
 3   Age          891 non-null    float64
 4   SibSp        891 non-null    int64  
 5   Parch        891 non-null    int64  
 6   Fare         891 non-null    float64
 7   Embarked_Q   891 non-null    bool   
 8   Embarked_S   891 non-null    bool   
dtypes: bool(2), float64(2), int64(5)
memory usage: 50.6 KB
None

Informations sur X_test:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Sex          418 non-null    int64  
 3

In [43]:
#Importer le modèle 
from sklearn.ensemble import RandomForestClassifier

#Créé le modèle Random Forest avec les arbres=estimators(mini modèle d'entrainement sur une partie aléatoire de la bdd) 
algo = RandomForestClassifier(n_estimators=500, random_state=42)

# Entraîner le modèle
algo.fit(X_train, y_train)

# Faire des prédictions sur les données de test
y_train_pred = algo.predict(X_train)
y_test_pred = algo.predict(X_test)

#crée un tableau result et renomer les colonnes
result = pd.concat([X_test["PassengerId"], pd.DataFrame(y_test_pred)], axis=1, ignore_index=True)
result.columns = ["PassengerId", "Survived"]
#affichage dans le notebook
result
#fichier csv
result.to_csv("/kaggle/working/result.csv", index=False)